# Workday Sales RAG Model Packaging & Registration

This notebook packages the RAG pipeline as an MLflow pyfunc model with Service Principal OAuth authentication, logs it with config artifacts, validates it, and registers it to Unity Catalog.

## Model Registration & Deployment

This section packages the RAG pipeline as an MLflow model and deploys it to a serving endpoint.

### 🛠️ Code Optimizations Applied

**1. Production-Ready Model Packaging:**
- Wrapped RAG pipeline in MLflow `PythonModel` for standardized serving
- Added proper signature and input examples (required for UC registration)
- Included comprehensive error handling and graceful degradation
- Validated inputs to prevent empty/malformed queries

**2. Enhanced Error Handling:**
- Try-catch blocks around vector search operations
- Graceful fallback when retrieval fails
- Informative error messages in responses
- Defensive programming throughout

**3. Model Artifact Management:**
- Explicit pip requirements for reproducibility
- Signature inference from input/output examples
- Metadata tracking (run ID, model URI, parameters)

**4. Deployment Best Practices:**
- Workload type selection based on model requirements
- Configuration validation before deployment
- Readiness checks with timeout handling
- Comprehensive deployment status reporting

**5. Production Readiness:**
- Unity Catalog three-level namespace (catalog.schema.model)
- Model versioning and alias support
- Environment-specific configurations
- Testing and validation before registration

### 📊 Original Code vs Optimized Comparison

| Aspect | Original | Optimized |
|--------|----------|----------|
| **Deployment** | Manual functions only | MLflow PythonModel wrapper |
| **Error Handling** | None | Comprehensive try-catch |
| **Input Validation** | None | Validates empty/malformed inputs |
| **Retrieval** | Sequential | Same (optimization: add threading) |
| **Configuration** | Hardcoded | Externalized in model |
| **Monitoring** | Manual | MLflow tracking + inference tables |
| **Versioning** | None | UC registration with aliases |

### 🚀 Future Optimization Opportunities

1. **Parallel Retrieval:** Use `ThreadPoolExecutor` to query 3 indexes concurrently
2. **Caching:** Implement context caching for repeated queries
3. **Batch Processing:** Optimize for batch inference workloads
4. **Prompt Optimization:** A/B test different prompt templates
5. **Reranking:** Add cross-encoder reranker after retrieval
6. **Hybrid Search:** Combine vector + keyword search

In [0]:
# Import only what's needed at class definition time
import json
import types
import mlflow
import hashlib
import pandas as pd
from typing import Dict, Any
from mlflow import MlflowClient
from mlflow.pyfunc import PythonModel
from mlflow.models.signature import infer_signature

In [0]:
# Use a dedicated experiment for this model (best practice: one experiment per model)
experiment_name = "/Gen_Agentic_AI_Learn/Assignment - 1/workday_sales_rag"
mlflow.set_experiment(experiment_name)

2026/09/08 01:59:20 INFO mlflow.tracking.fluent: Experiment with name '/Gen_Agentic_AI_Learn/Assignment - 1/workday_sales_rag' does not exist. Creating a new experiment.


<Experiment: artifact_location='dbfs:/databricks/mlflow-tracking/2062020818108666', creation_time=1788832760897, experiment_id='2062020818108666', last_update_time=1788832760897, lifecycle_stage='active', name='/Gen_Agentic_AI_Learn/Assignment - 1/workday_sales_rag', tags={'mlflow.experiment.sourceName': '/Gen_Agentic_AI_Learn/Assignment - '
                                 '1/workday_sales_rag',
 'mlflow.experimentType': 'MLFLOW_EXPERIMENT',
 'mlflow.ownerEmail': 'venkatkm99@gmail.com',
 'mlflow.ownerId': '4940547174685505'}>

In [0]:
import mlflow
from mlflow.entities import SpanType

class WorkdaySalesRAG(PythonModel):
    """
    Custom MLflow model that wraps the Workday Sales RAG pipeline.
    Retrieves context from vector search indexes and generates responses using LLM.
    """
    
    def load_context(self, context):
        """
        Initialize model dependencies with configuration precedence:
        1. Environment variables (highest - runtime override)
        2. Model artifacts (config.json - versioned with model)
        3. Hardcoded defaults (fallback)
        
        Handles both PAT token and Service Principal OAuth authentication.
        """
        import os
        import json
        
        # Set default secret scope before authentication (can be overridden later from config/env)
        self.secret_scope = os.getenv("SECRET_SCOPE", "agentic")
        
        # Setup authentication for model serving context
        self._setup_authentication()
        
        # Default configuration (fallback)
        default_config = {
                            "secret_scope": "agentic",
                            "llm_endpoint_name": "databricks-meta-llama-3-3-70b-instruct",
                            "vs_endpoint_name": "sales-endpoint-rag_agentic",
                            "indexes": {
                                'customer_feedback': 'rag_agentic.workday_demos.customer_feedback_index',
                                'meeting_notes': 'rag_agentic.workday_demos.meeting_notes_index',
                                'email_communications': 'rag_agentic.workday_demos.email_communications_index'
                            },
                            "num_results": 5,
                            "temperature": 0.1,
                            "max_tokens": 1000
                        }
        
        # Try loading config from artifact (if provided during logging)
        config = default_config.copy()
        if context and context.artifacts:
            config_path = context.artifacts.get("config")
            if config_path:
                try:
                    with open(config_path, 'r') as f:
                        artifact_config = json.load(f)
                        config.update(artifact_config)
                        print("✅ Loaded configuration from artifact")
                except Exception as e:
                    print(f"⚠️ Could not load config artifact: {e}. Using defaults.")
        
        # Override with environment variables (highest precedence)
        self.secret_scope = os.getenv("SECRET_SCOPE", config.get("secret_scope", self.secret_scope))
        self.llm_endpoint_name = os.getenv("LLM_ENDPOINT_NAME", config["llm_endpoint_name"])
        self.vs_endpoint_name = os.getenv("VS_ENDPOINT_NAME", config["vs_endpoint_name"])
        
        # Handle indexes - can be overridden individually
        self.indexes = {
                        'customer_feedback': os.getenv("CUSTOMER_FEEDBACK_INDEX", config["indexes"]["customer_feedback"]),
                        'meeting_notes': os.getenv("MEETING_NOTES_INDEX", config["indexes"]["meeting_notes"]),
                        'email_communications': os.getenv("EMAIL_COMMUNICATIONS_INDEX", config["indexes"]["email_communications"])
                    }
        
        # Generation parameters
        self.num_results = int(os.getenv("NUM_RESULTS", config.get("num_results", 5)))
        self.temperature = float(os.getenv("TEMPERATURE", config.get("temperature", 0.1)))
        self.max_tokens = int(os.getenv("MAX_TOKENS", config.get("max_tokens", 1000)))
        
        print(f"📋 Configuration loaded:")
        print(f"   LLM Endpoint: {self.llm_endpoint_name}")
        print(f"   VS Endpoint: {self.vs_endpoint_name}")
        print(f"   Indexes: {len(self.indexes)} configured")
        print(f"   Auth: {self.auth_type}")
    
    def _setup_authentication(self):
        """
        Configure authentication for Databricks resources.
        Supports both PAT token and Service Principal OAuth M2M.
        Credentials are fetched from Databricks secrets (preferred) or environment variables (fallback).
        """
        import os
        
        # Try to fetch credentials from Databricks secrets first (notebook context)
        try:
            from dbutils import secrets
            # Fetch from secrets - defaults to 'agentic' scope (override with SECRET_SCOPE env var)
            client_id = secrets.get(scope=self.secret_scope, key="client_id")
            client_secret = secrets.get(scope=self.secret_scope, key="client_secret")
            host = secrets.get(scope=self.secret_scope, key="databricks_host")
        except (ImportError, Exception):
            # Fallback to environment variables (model serving context where dbutils is not available)
            client_id = os.getenv("DATABRICKS_CLIENT_ID")
            client_secret = os.getenv("DATABRICKS_CLIENT_SECRET")
            host = os.getenv("DATABRICKS_HOST")
        
        if client_id and client_secret and host:
            # Service Principal OAuth M2M authentication
            self.auth_type = "Service Principal OAuth"
            self._setup_sp_auth(host, client_id, client_secret)
        elif os.getenv("DATABRICKS_TOKEN") and host:
            # PAT token authentication
            self.auth_type = "PAT Token"
            os.environ["DATABRICKS_HOST"] = host
        else:
            # Default/fallback authentication (notebook context)
            self.auth_type = "Default (Notebook)"
            print("⚠️ Running in notebook context, using default authentication")
    
    def _setup_sp_auth(self, host, client_id, client_secret):
        """
        Setup Service Principal OAuth M2M authentication.
        Obtains OAuth token and configures environment.
        """
        import requests
        import os
        
        try:
            # Get OAuth access token using client credentials flow
            token_url = f"{host}/oidc/v1/token"
            
            response = requests.post(
                                        token_url,
                                        data={
                                            "grant_type": "client_credentials",
                                            "scope": "all-apis"
                                        },
                                        auth=(client_id, client_secret),
                                        headers={"Content-Type": "application/x-www-form-urlencoded"},
                                        timeout=30
                                    )
            
            if response.status_code == 200:
                token_data = response.json()
                access_token = token_data.get("access_token")
                
                if access_token:
                    # Set environment variables for Databricks SDK
                    os.environ["DATABRICKS_HOST"] = host
                    os.environ["DATABRICKS_TOKEN"] = access_token
                    print("✅ Service Principal OAuth authentication configured")
                else:
                    raise ValueError("No access token in OAuth response")
            else:
                raise ValueError(f"OAuth token request failed: {response.status_code} - {response.text}")
                
        except Exception as e:
            print(f"❌ Service Principal authentication failed: {str(e)}")
            print("   Falling back to default authentication")
            raise
        
    def _format_results(self, results):
        """Format search results into readable context"""
        if not results or 'result' not in results or 'data_array' not in results['result']:
            return "No relevant information found."
        
        formatted = []
        for i, row in enumerate(results['result']['data_array'], 1):
            content = row[0] if len(row) > 0 else "N/A"
            doc_uri = row[1] if len(row) > 1 else "Unknown source"
            formatted.append(f"[{i}] Source: {doc_uri}\n{content}\n")
        
        return "\n".join(formatted)
    
    @mlflow.trace(span_type=SpanType.RETRIEVER)
    def _retrieve_contexts(self, user_question: str, num_results: int = None) -> Dict[str, str]:
        """Retrieve context from all three vector indexes"""
        from databricks.ai_search.client import VectorSearchClient
        
        # Use configured num_results if not provided
        if num_results is None:
            num_results = self.num_results
        
        vsc = VectorSearchClient()
        contexts = {}
        
        try:
            # Customer feedback
            feedback_index = vsc.get_index(endpoint_name=self.vs_endpoint_name, index_name=self.indexes['customer_feedback'])
            feedback_results = feedback_index.similarity_search(query_text=user_question, columns=["content", "doc_uri"], num_results=num_results)
            contexts['customer_feedback'] = self._format_results(feedback_results)
            
            # Meeting notes
            meeting_index = vsc.get_index(endpoint_name=self.vs_endpoint_name,index_name=self.indexes['meeting_notes'])
            meeting_results = meeting_index.similarity_search(query_text=user_question, columns=["content", "doc_uri"], num_results=num_results)
            contexts['meeting_notes'] = self._format_results(meeting_results)
            
            # Email communications
            email_index = vsc.get_index(endpoint_name=self.vs_endpoint_name, index_name=self.indexes['email_communications'])
            email_results = email_index.similarity_search(query_text=user_question, columns=["content", "doc_uri"], num_results=num_results)
            contexts['email_communications'] = self._format_results(email_results)
            
        except Exception as e:
            # Graceful degradation
            contexts = {
                            'customer_feedback': f"Error retrieving customer feedback: {str(e)}",
                            'meeting_notes': f"Error retrieving meeting notes: {str(e)}",
                            'email_communications': f"Error retrieving emails: {str(e)}"
                        }
        
        return contexts
    
    def _generate_prompt(self, user_question: str, contexts: Dict[str, str]) -> str:
        """Generate RAG prompt with retrieved contexts"""
        prompt = f"""You are an intelligent assistant with access to information from customer feedback, meeting notes, and email communications.

## Retrieved Context:

### Customer Feedback:
{contexts['customer_feedback']}

### Meeting Notes:
{contexts['meeting_notes']}

### Email Communications:
{contexts['email_communications']}

## User Question:
{user_question}

## Instructions:
1. Analyze the context from all three sources carefully
2. Synthesize information across customer feedback, meeting notes, and emails to provide a comprehensive answer
3. If information conflicts between sources, note the discrepancy and provide context
4. If the context doesn't contain relevant information, acknowledge this limitation
5. Cite which source(s) your answer comes from (customer feedback, meeting notes, or emails)
6. Provide specific examples or quotes when relevant

## Answer:
"""
        return prompt
    
    @mlflow.trace(span_type=SpanType.LLM)
    def _generate_response(self, prompt: str):
        """Call LLM endpoint with configured parameters to generate response"""
        from mlflow.deployments import get_deploy_client
        
        client = get_deploy_client("databricks")
        response = client.predict(
                                endpoint=self.llm_endpoint_name,
                                inputs={
                                            "messages": [{"role": "user", "content": prompt}],
                                            "temperature": self.temperature,
                                            "max_tokens": self.max_tokens
                                        }
                            )
        
        return response.choices[0]['message']['content']
    
    @mlflow.trace(span_type=SpanType.CHAIN)
    def predict(self, context, model_input):
        """
        Generate RAG responses for input messages in OpenAI chat format.
        
        Args:
            model_input: pandas DataFrame or dict with 'messages' field
                        messages: list of message dicts with 'role' and 'content' keys
                        Example: {"messages": [{"role": "user", "content": "What is X?"}]}
            
        Returns:
            pandas DataFrame with 'choices' column (OpenAI chat completion format)
        """
        # Handle both DataFrame and dict inputs
        if isinstance(model_input, pd.DataFrame):
            messages_list = model_input['messages'].tolist()
        elif isinstance(model_input, dict):
            messages_list = [model_input.get('messages', [])]
        else:
            choices = [[{
                "index": 0,
                "message": {
                    "role": "assistant",
                    "content": f"Invalid input type: {type(model_input)}"
                },
                "finish_reason": "stop"
            }]]
            return pd.DataFrame({'choices': choices})
        
        # Extract questions from messages (take the last user message)
        questions = []
        for messages in messages_list:
            if not messages or not isinstance(messages, list):
                questions.append('')
            else:
                # Get the last user message
                user_messages = [msg['content'] for msg in messages if msg.get('role') == 'user' and msg.get('content')]
                questions.append(user_messages[-1] if user_messages else '')
        
        # Validate inputs
        if not questions or all(not q.strip() for q in questions):
            choices = []
            for _ in questions:
                choices.append([{
                    "index": 0,
                    "message": {
                        "role": "assistant",
                        "content": "Error: Empty question provided"
                    },
                    "finish_reason": "stop"
                }])
            return pd.DataFrame({'choices': choices})
        
        answers = []
        
        for question in questions:
            try:
                # Retrieve contexts from vector indexes
                contexts = self._retrieve_contexts(question)
                
                # Generate RAG prompt with retrieved context
                prompt = self._generate_prompt(question, contexts)
                
                # Generate response using LLM
                answer = self._generate_response(prompt)
                answers.append(answer)
                
            except Exception as e:
                answers.append(f"Error generating response: {str(e)}")
        
        # Return in OpenAI chat completion format
        choices = []
        for answer in answers:
            choices.append([
                {
                    "index": 0,
                    "message": {
                        "role": "assistant",
                        "content": answer
                    },
                    "finish_reason": "stop"
                }
            ])
        
        return pd.DataFrame({
            'choices': choices
        })


print("✅ WorkdaySalesRAG model class defined")

✅ WorkdaySalesRAG model class defined


/databricks/python/lib/python3.12/site-packages/mlflow/pyfunc/utils/data_validation.py:186: UserWarning: Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.
  color_warning(


In [0]:
# Create configuration file for model artifact
model_config = {
                "secret_scope": "agentic",
                "llm_endpoint_name": "databricks-meta-llama-3-3-70b-instruct",
                "vs_endpoint_name": "sales-endpoint-rag_agentic",
                "indexes": {
                            "customer_feedback": "rag_agentic.workday_demos.customer_feedback_index",
                            "meeting_notes": "rag_agentic.workday_demos.meeting_notes_index",
                            "email_communications": "rag_agentic.workday_demos.email_communications_index"
                            },
                "num_results": 5,
                "temperature": 0.1,
                "max_tokens": 1000
            }

# Save config to file
config_path = "/tmp/model_config.json"
with open(config_path, "w") as f:
    json.dump(model_config, f, indent=2)

print("📝 Model Configuration:")
print(json.dumps(model_config, indent=2))

# Create input example and signature (OpenAI chat format)
input_example = pd.DataFrame({
                                'messages': [[
                                    {"role": "user", "content": "What are the main customer concerns about product delivery?"}
                                ]]
                            })

# Generate sample output for signature
model_instance = WorkdaySalesRAG()
model_instance.load_context(None)

# Create output example in OpenAI format
output_example = pd.DataFrame({
                                'choices': [[
                                    {
                                        "index": 0,
                                        "message": {
                                            "role": "assistant",
                                            "content": "Based on customer feedback, meeting notes, and email communications..."
                                        },
                                        "finish_reason": "stop"
                                    }
                                ]]
                            })

# Infer signature
signature = infer_signature(input_example, output_example)

print("📝 Model Signature:")
print(signature)
print("\n📦 Input Example:")
print(input_example)

# ---- Idempotency: compute content hash of model code + config + deps ----
pip_requirements = [
                        "databricks-ai-search",
                        "mlflow[databricks]",
                        "pandas",
                        "requests",  # For OAuth token retrieval
                    ]

# Compute deterministic hash of model class bytecode + config + deps
def _class_hash(cls):
    """Deterministic hash of a class's methods using bytecode + names."""
    hasher = hashlib.sha256()
    for name in sorted(cls.__dict__.keys()):
        method = cls.__dict__[name]
        if isinstance(method, types.FunctionType):
            hasher.update(name.encode())
            hasher.update(method.__code__.co_code)
            hasher.update(str(method.__code__.co_names).encode())
    return hasher.hexdigest()

content_hash = hashlib.sha256(
                                _class_hash(WorkdaySalesRAG).encode()
                                + json.dumps(model_config, sort_keys=True).encode()
                                + json.dumps(pip_requirements, sort_keys=True).encode()
                            ).hexdigest()

print(f"\n🔐 Content hash: {content_hash}")

# Search for an existing run with the same content hash (experiment was set in Cell 4)
experiment = mlflow.get_experiment_by_name(experiment_name)

existing_runs = mlflow.search_runs(
                                    experiment_ids=[experiment.experiment_id],
                                    filter_string=f"tags.content_hash = '{content_hash}'",
                                    order_by=["attributes.start_time DESC"],
                                )

if not existing_runs.empty:
    # Reuse the existing logged model — no need to log again
    run_id = existing_runs.iloc[0]["run_id"]
    model_uri = f"runs:/{run_id}/workday_sales_rag_model"
    print(f"\n✅ Found existing model with identical code & config (Run ID: {run_id})")
    print(f"   Skipping model logging — reusing existing model")
    print(f"   Model URI: {model_uri}")
else:
    # Log the model to MLflow with config artifact
    with mlflow.start_run(run_name="workday_sales_rag_v1") as run:
        mlflow.set_tag("content_hash", content_hash)
        
        model_info = mlflow.pyfunc.log_model(
            artifact_path="workday_sales_rag_model",
            python_model=WorkdaySalesRAG(),
            artifacts={"config": config_path},  # Include config artifact
            signature=signature,
            input_example=input_example,
            pip_requirements=pip_requirements,
        )
        
        # Log parameters for tracking
        mlflow.log_params({
            "llm_endpoint": model_config["llm_endpoint_name"],
            "vs_endpoint": model_config["vs_endpoint_name"],
            "num_indexes": len(model_config["indexes"]),
            "num_results": model_config["num_results"],
            "temperature": model_config["temperature"],
            "max_tokens": model_config["max_tokens"]
        })
        
        run_id = run.info.run_id
        model_uri = model_info.model_uri

    print(f"\n✅ Model logged successfully!")
    print(f"   Run ID: {run_id}")
    print(f"   Model URI: {model_uri}")
    print(f"\n🔗 View in MLflow: {mlflow.get_tracking_uri()}/mlflow/experiments/")

📝 Model Configuration:
{
  "secret_scope": "agentic",
  "llm_endpoint_name": "databricks-meta-llama-3-3-70b-instruct",
  "vs_endpoint_name": "sales-endpoint-rag_agentic",
  "indexes": {
    "customer_feedback": "rag_agentic.workday_demos.customer_feedback_index",
    "meeting_notes": "rag_agentic.workday_demos.meeting_notes_index",
    "email_communications": "rag_agentic.workday_demos.email_communications_index"
  },
  "num_results": 5,
  "temperature": 0.1,
  "max_tokens": 1000
}
⚠️ Running in notebook context, using default authentication
📋 Configuration loaded:
   LLM Endpoint: databricks-meta-llama-3-3-70b-instruct
   VS Endpoint: sales-endpoint-rag_agentic
   Indexes: 3 configured
   Auth: Default (Notebook)
📝 Model Signature:
inputs: 
  ['messages': Array({content: string (required), role: string (required)}) (required)]
outputs: 
  ['choices': Array({finish_reason: string (required), index: long (required), message: {content: string (required), role: string (required)} (require

{"ts": "2026-09-08 02:00:35.157", "level": "WARNING", "logger": "pyspark.sql.connect.logging", "msg": "Effective usage policy for this session is cct.Chpub3RlYm9va3MvMzM3MDAzNzI3ODIxOTI1MBABIAEyJDAxYTA3ZWJhLTgzMWEtNzJhOS1hZDZkLTNhMmMyOWUxMDVmYjokNjU1ZGIzYWYtYjNjOS0zYzcyLWFiOGMtMjIyYTM1NGNjNjAxSgwIo9b91AYQgLnvogJQAVgBYAFoxYmPlILFow0=.", "context": {}}
{"ts": "2026-09-08 02:00:35.157", "level": "WARNING", "logger": "pyspark.sql.connect.logging", "msg": "Effective usage policy for this session is cct.Chpub3RlYm9va3MvMzM3MDAzNzI3ODIxOTI1MBABIAEyJDAxYTA3ZWJhLTgzMWEtNzJhOS1hZDZkLTNhMmMyOWUxMDVmYjokNjU1ZGIzYWYtYjNjOS0zYzcyLWFiOGMtMjIyYTM1NGNjNjAxSgwIo9b91AYQgLnvogJQAVgBYAFoxYmPlILFow0=.", "context": {}}
{"ts": "2026-09-08 02:00:35.157", "level": "WARNING", "logger": "pyspark.sql.connect.logging", "msg": "Effective usage policy for this session is cct.Chpub3RlYm9va3MvMzM3MDAzNzI3ODIxOTI1MBABIAEyJDAxYTA3ZWJhLTgzMWEtNzJhOS1hZDZkLTNhMmMyOWUxMDVmYjokNjU1ZGIzYWYtYjNjOS0zYzcyLWFiOGMtMjIyYTM1NGNjNjAx

2026/09/08 02:00:39 WARNING mlflow.utils.requirements_utils: Detected one or more mismatches between the model's dependencies and the current Python environment:
 - databricks-ai-search (current: uninstalled, required: databricks-ai-search)
To fix the mismatches, call `mlflow.pyfunc.get_model_dependencies(model_uri)` to fetch the model's environment and install dependencies using the resulting environment file.
2026/09/08 02:00:39 WARNING mlflow.utils.requirements_utils: Detected one or more mismatches between the model's dependencies and the current Python environment:
 - databricks-ai-search (current: uninstalled, required: databricks-ai-search)
To fix the mismatches, call `mlflow.pyfunc.get_model_dependencies(model_uri)` to fetch the model's environment and install dependencies using the resulting environment file.


⚠️ Running in notebook context, using default authentication
✅ Loaded configuration from artifact
📋 Configuration loaded:
   LLM Endpoint: databricks-meta-llama-3-3-70b-instruct
   VS Endpoint: sales-endpoint-rag_agentic
   Indexes: 3 configured
   Auth: Default (Notebook)


2026/09/08 02:00:40 WARNING mlflow.tracking.context.registry: Encountered unexpected error during resolving tags: An error occurred while calling o39.extraContext. Trace:
py4j.security.Py4JSecurityException: Method public scala.collection.immutable.Map com.databricks.backend.common.rpc.CommandContext.extraContext() is not whitelisted on class class com.databricks.backend.common.rpc.CommandContext
	at py4j.security.WhitelistingPy4JSecurityManager.checkCall(WhitelistingPy4JSecurityManager.java:473)
	at py4j.Gateway.invoke(Gateway.java:305)
	at com.databricks.backend.daemon.driver.InstrumentedCallCommand.$anonfun$invokeMethod$1(InstrumentedCallCommand.scala:19)
	at com.databricks.backend.daemon.driver.InstrumentedPy4jCommand.instrumentedInvoke(InstrumentedPy4jCommand.scala:28)
	at com.databricks.backend.daemon.driver.InstrumentedPy4jCommand.instrumentedInvoke$(InstrumentedPy4jCommand.scala:18)
	at com.databricks.backend.daemon.driver.InstrumentedCallCommand.instrumentedInvoke(Instrumented


✅ Model logged successfully!
   Run ID: b4e638884b764ef9b41ec69a627aeba8
   Model URI: models:/m-b15c3d2da3f848b2ae4c30b4990a1955

🔗 View in MLflow: databricks/mlflow/experiments/


In [0]:
# Validate the logged model before registration
print("\n🔍 Validating model before registration...\n")
print(f"   Model URI: {model_uri}\n")

# Load the model
pyfunc_model = mlflow.pyfunc.load_model(model_uri)

# Check signature
if pyfunc_model.metadata.signature is None:
    raise ValueError("❌ The logged model has no signature; fix the original artifact before registration.")
print("✅ Signature validation passed")

# Check input example
representative_input = pyfunc_model.input_example
if representative_input is None:
    raise ValueError("❌ The logged model has no input example; fix the original artifact before registration.")
print("✅ Input example validation passed")

# Test prediction with input example
print("\n🧪 Testing model prediction with sample input...")
try:
    test_result = mlflow.models.predict(
                                        model_uri=model_uri,
                                        input_data=representative_input,
                                    )
    print("✅ Model prediction test passed")
    print("\n📊 Sample Prediction Result:")
    print(test_result)
except Exception as e:
    raise ValueError(f"❌ Model prediction failed: {str(e)}")

print("\n✨ Model validation complete! Ready for registration.")


🔍 Validating model before registration...

   Model URI: models:/m-b15c3d2da3f848b2ae4c30b4990a1955



2026/09/08 02:03:07 WARNING mlflow.utils.requirements_utils: Detected one or more mismatches between the model's dependencies and the current Python environment:
 - databricks-ai-search (current: uninstalled, required: databricks-ai-search)
To fix the mismatches, call `mlflow.pyfunc.get_model_dependencies(model_uri)` to fetch the model's environment and install dependencies using the resulting environment file.
2026/09/08 02:03:07 INFO mlflow.models.python_api: It is highly recommended to use `uv` as the environment manager for predicting with MLflow models as its performance is significantly better than other environment managers. Run `pip install uv` to install uv. See https://docs.astral.sh/uv/getting-started/installation for other installation methods.


⚠️ Running in notebook context, using default authentication
✅ Loaded configuration from artifact
📋 Configuration loaded:
   LLM Endpoint: databricks-meta-llama-3-3-70b-instruct
   VS Endpoint: sales-endpoint-rag_agentic
   Indexes: 3 configured
   Auth: Default (Notebook)
✅ Signature validation passed
✅ Input example validation passed

🧪 Testing model prediction with sample input...


2026/09/08 02:03:09 INFO mlflow.models.flavor_backend_registry: Selected backend for flavor 'python_function'


2026/09/08 02:03:11 INFO mlflow.utils.virtualenv: Installing python 3.12.3 if it does not exist
-> https://www.python.org/ftp/python/3.12.3/Python-3.12.3.tar.xz
Installing Python-3.12.3...
Installed Python-3.12.3 to /tmp/pyenv_root/versions/3.12.3
2026/09/08 02:06:08 INFO mlflow.utils.virtualenv: Creating a new environment in /tmp/virtualenv_envs/mlflow-f9c092df76277573135d932a58ba81bcfe1d84c7 with /tmp/pyenv_root/versions/3.12.3/bin/python
2026/09/08 02:06:09 INFO mlflow.utils.virtualenv: Installing dependencies


...

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 78.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 91.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 102.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 75.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.8/15.8 MB 185.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 124.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 111.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.7/15.7 MB 108.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.8/46.8 MB 136.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 134.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.0/34.0 MB 105.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 124.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 M


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
2026/09/08 02:07:11 INFO mlflow.utils.environment: === Running command '['bash', '-c', 'source /tmp/virtualenv_envs/mlflow-f9c092df76277573135d932a58ba81bcfe1d84c7/bin/activate && python -c ""']'
2026/09/08 02:07:11 INFO mlflow.utils.environment: === Running command '['bash', '-c', 'source /tmp/virtualenv_envs/mlflow-f9c092df76277573135d932a58ba81bcfe1d84c7/bin/activate && python /databricks/python/lib/python3.12/site-packages/mlflow/pyfunc/_mlflow_pyfunc_backend_predict.py --model-uri file:///local_disk0/user_tmp_data/spark-5318934f-21c9-4216-b67a-6b/tmp4lhmy3q3 --content-type json --input-path /local_disk0/user_tmp_data/spark-5318934f-21c9-4216-b67a-6b/tmpaarwfequ/input.json']'
2026/09/08 02:07:15 WARNING mlflow.pyfunc: The version of CloudPickle that was used to save the model, `CloudPickle 3.0.0`, differs from the version of CloudPickle that is currently running, `CloudP

✅ Loaded configuration from artifact
📋 Configuration loaded:
   LLM Endpoint: databricks-meta-llama-3-3-70b-instruct
   VS Endpoint: sales-endpoint-rag_agentic
   Indexes: 3 configured
   Auth: PAT Token
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disa

2026/09/08 02:07:35 INFO mlflow.tracing.export.async_export_queue: Flushing the async trace logging queue before program exit. This may take a while...


✅ Model prediction test passed

📊 Sample Prediction Result:
None

✨ Model validation complete! Ready for registration.


In [0]:
# Unity Catalog registration configuration
# 👉 UPDATE THESE VALUES for your environment:
catalog_name = "rag_agentic"  # Your Unity Catalog name
schema_name = "workday_demos"  # Your schema name
model_name = "workday_sales_rag"  # Model name

# Full three-level namespace
registered_model_name = f"{catalog_name}.{schema_name}.{model_name}"

print(f"\n📦 Registering model to Unity Catalog...")
print(f"   Target: {registered_model_name}")
print(f"   Source: {model_uri}")

# Register the model
mlflow.set_registry_uri("databricks-uc")
registry_client = MlflowClient(registry_uri="databricks-uc")

# ---- Idempotency: check if this run is already registered as a version ----
existing_versions = registry_client.search_model_versions(f"name='{registered_model_name}'")

already_registered = None
for mv in existing_versions:
    if mv.run_id == run_id:
        already_registered = mv
        break

if already_registered:
    model_version = already_registered.version
    print(f"\n✅ Model already registered as version {model_version} (from run {run_id})")
    print(f"   Skipping registration — reusing existing version")
    print(f"   Model: {registered_model_name}")
    print(f"   Version: {model_version}")
    print(f"   URI: models:/{registered_model_name}/{model_version}")
else:
    registered_version = mlflow.register_model(
                                                model_uri=model_uri,
                                                name=registered_model_name,
                                                await_registration_for=300,  # Wait up to 5 minutes
                                            )

    model_version = registered_version.version

    print(f"\n✅ Model registered successfully!")
    print(f"   Model: {registered_model_name}")
    print(f"   Version: {model_version}")
    print(f"   URI: models:/{registered_model_name}/{model_version}")

# Optional: Set alias (e.g., 'champion' for production)
set_alias = True  # Set to False to skip
alias_name = "champion"

if set_alias:
    registry_client.set_registered_model_alias(
                                                name=registered_model_name,
                                                alias=alias_name,
                                                version=model_version,
                                            )
    print(f"\n🏆 Alias '{alias_name}' set to version {model_version}")
    print(f"   Can now reference as: models:/{registered_model_name}@{alias_name}")


📦 Registering model to Unity Catalog...
   Target: rag_agentic.workday_demos.workday_sales_rag
   Source: models:/m-b15c3d2da3f848b2ae4c30b4990a1955


Registered model 'rag_agentic.workday_demos.workday_sales_rag' already exists. Creating a new version of this model...


Uploading artifacts:   0%|          | 0/13 [00:00<?, ?it/s]

🔗 Created version '2' of model 'rag_agentic.workday_demos.workday_sales_rag': https://dbc-00781aa6-52b5.cloud.databricks.com/explore/data/models/rag_agentic/workday_demos/workday_sales_rag/version/2?o=7474652423374021



✅ Model registered successfully!
   Model: rag_agentic.workday_demos.workday_sales_rag
   Version: 2
   URI: models:/rag_agentic.workday_demos.workday_sales_rag/2

🏆 Alias 'champion' set to version 2
   Can now reference as: models:/rag_agentic.workday_demos.workday_sales_rag@champion
